# data cleaning

our question is whether weather affects how much people buy online and whether they return stuff later

we have olist ecommerce data (2016-2018) and brazil weather station data (2000-2021)

this notebook just gets rid of all the stuff we dont need

Side Note: used ai tools like claude code when i couldn't do it by myself

In [1]:
import pandas as pd
import numpy as np
import os

# make a folder to save the cleaned files
os.makedirs('cleaned', exist_ok=True)

## orders

In [2]:
orders = pd.read_csv('olist_orders_dataset.csv')
print(orders.shape)
orders.head(3)

(99441, 8)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [3]:
# we only need order id, customer id, the status and when it was bought
# drop the delivery/approval timestamps we dont care about those
orders = orders[['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp']]

orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

# keep only 2016-2018 to match the weather data we have
orders = orders[orders['order_purchase_timestamp'] >= '2016-01-01']
orders = orders[orders['order_purchase_timestamp'] < '2019-01-01']

# add a date column and hour column, will be useful later
orders['purchase_date'] = orders['order_purchase_timestamp'].dt.date
orders['purchase_hour'] = orders['order_purchase_timestamp'].dt.hour

# mark cancelled and unavailable orders as returned (for the guilt question)
orders['is_returned'] = orders['order_status'].isin(['canceled', 'unavailable']).astype(int)

print(orders.shape)
print(orders['order_status'].value_counts())

(99441, 7)
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [4]:
orders.to_csv('cleaned/orders.csv', index=False)
print('done')

done


## customers

In [5]:
customers = pd.read_csv('olist_customers_dataset.csv')
print(customers.shape)
customers.head(3)

(99441, 5)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


In [6]:
# only keep customers who actually made an order in our date range
customers = customers[customers['customer_id'].isin(orders['customer_id'])]

# all 5 columns are useful (zip code links to location, state helps match to weather)
print(customers.shape)
print(customers.isnull().sum())

(99441, 5)
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


In [7]:
customers.to_csv('cleaned/customers.csv', index=False)
print('done')

done


## order items - how much was spent per order

In [8]:
items = pd.read_csv('olist_order_items_dataset.csv')
print(items.shape)
items.head(3)

(112650, 7)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


In [9]:
# only keep orders from our date range
items = items[items['order_id'].isin(orders['order_id'])]

# we dont need individual item rows, just group by order and get totals
# drop seller_id, shipping_limit_date - not relevant
order_spend = items.groupby('order_id').agg(
    item_count    = ('order_item_id', 'max'),
    total_price   = ('price', 'sum'),
    total_freight = ('freight_value', 'sum')
).reset_index()

print(order_spend.shape)
order_spend.head(3)

(98666, 4)


,order_id,item_count,total_price,total_freight
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,199.0,17.87


In [10]:
order_spend.to_csv('cleaned/order_spend.csv', index=False)
print('done')

done


## payments

In [11]:
payments = pd.read_csv('olist_order_payments_dataset.csv')
print(payments.shape)
payments.head(3)

(103886, 5)


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


In [12]:
payments = payments[payments['order_id'].isin(orders['order_id'])]

# group by order to get total paid and number of installments
# drop payment_sequential, we dont need individual installment rows
order_payments = payments.groupby('order_id').agg(
    total_payment_value  = ('payment_value', 'sum'),
    payment_installments = ('payment_installments', 'max')
).reset_index()

# get the main payment type per order (whichever had the highest value)
top_payment = payments.sort_values('payment_value', ascending=False).drop_duplicates('order_id')
order_payments = order_payments.merge(top_payment[['order_id', 'payment_type']], on='order_id', how='left')

print(order_payments.shape)
order_payments.head(3)

(99440, 4)

,order_id,total_payment_value,payment_installments,payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,5,credit_card


In [13]:
order_payments.to_csv('cleaned/order_payments.csv', index=False)
print('done')

done


## products - just need the category names in english

In [14]:
products = pd.read_csv('olist_products_dataset.csv')
translation = pd.read_csv('product_category_name_translation.csv')

print(products.shape)
products.head(3)

(32951, 9)


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0


In [15]:
# drop all the physical dimension stuff, we only care about category
products = products[['product_id', 'product_category_name']]

# add the english translation
products = products.merge(translation, on='product_category_name', how='left')

# use english name, fall back to portuguese if no translation
products['category'] = products['product_category_name_english'].fillna(products['product_category_name'])
products = products[['product_id', 'category']]

# drop the few rows with no category at all
products = products.dropna(subset=['category'])

print(products.shape)
print(products.isnull().sum())

(32341, 2)
product_id    0
category      0
dtype: int64


In [16]:
# also attach the main category to each order
# join items back to products to get category per item, then pick most common one per order
items_with_cat = items[['order_id', 'product_id']].merge(products, on='product_id', how='left')

order_category = items_with_cat.groupby('order_id')['category'].agg(
    lambda x: x.mode()[0] if len(x.mode()) > 0 else np.nan
).reset_index()
order_category.columns = ['order_id', 'main_category']

# drop orders where category is missing
order_category = order_category.dropna(subset=['main_category'])

print(order_category.shape)
order_category.head(3)

(97277, 2)


,order_id,main_category
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,pet_shop
2,000229ec398224ef6ca0657da4fc703e,furniture_decor


In [17]:
products.to_csv('cleaned/products.csv', index=False)
order_category.to_csv('cleaned/order_category.csv', index=False)
print('done')

done


## geolocation - zip code to coordinates

we need this to match customers to their nearest weather station

In [18]:
geo = pd.read_csv('olist_geolocation_dataset.csv')
print(geo.shape)
geo.head(3)

(1000163, 5)


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP


In [19]:
# only keep zips that our customers actually use
geo = geo[geo['geolocation_zip_code_prefix'].isin(customers['customer_zip_code_prefix'])]

# drop city column, we dont need it
# there are loads of duplicate rows per zip so average the lat/lng
geo = geo.groupby('geolocation_zip_code_prefix').agg(
    lat   = ('geolocation_lat', 'mean'),
    lng   = ('geolocation_lng', 'mean'),
    state = ('geolocation_state', lambda x: x.mode()[0])
).reset_index()

geo = geo.rename(columns={'geolocation_zip_code_prefix': 'zip_code_prefix'})

print(geo.shape)
print(geo.isnull().sum())

(14837, 4)
zip_code_prefix    0
lat                0
lng                0
state              0
dtype: int64


In [20]:
geo.to_csv('cleaned/geolocation.csv', index=False)
print('done')

done


## weather stations - just need location of each station

In [21]:
stations = pd.read_csv('automatic_stations_codes_2000_2021.csv', sep=';')
print(stations.shape)
stations.head(3)

(612, 7)


,REGIAO,UF,ESTACAO,CODIGO,LATITUDE,LONGITUDE,ALTITUDE
0,N,PA,SANTA MARIA DAS BARREIRAS,A256,-8.729722,-49.856389,165.00
1,SE,SP,CRIOSFERA,C891,-84.000000,-79.494167,1285.00
2,CO,DF,BRASILIA,A001,-15.789444,-47.925833,1159.54


In [22]:
# only keep station id, coordinates and state
# drop region, name, altitude
stations = stations[['CODIGO', 'LATITUDE', 'LONGITUDE', 'UF']]
stations.columns = ['station_id', 'lat', 'lng', 'state']

# drop rows with missing coordinates
stations = stations.dropna(subset=['lat', 'lng'])

# there is one station (CRIOSFERA) with coordinates in antarctica which is clearly wrong, remove it
stations = stations[stations['lat'] > -60]

print(stations.shape)
stations.head(3)

(611, 4)


,station_id,lat,lng,state
0,A256,-8.729722,-49.856389,PA
2,A001,-15.789444,-47.925833,DF
3,A401,-13.016667,-38.516667,BA


In [23]:
stations.to_csv('cleaned/weather_stations.csv', index=False)
print('done')

done


## weather data

this file is huge (~60 million rows). we only need 2016-2018 and only temperature, rain and humidity.
we also turn hourly data into daily averages because we dont need per-hour precision.

reading it in chunks for the sake of my pc

In [24]:
# columns we want to keep (the portuguese names in the file)
cols_we_want = [
    'ESTACAO',
    'DATA (YYYY-MM-DD)',
    'PRECIPITACAO TOTAL HORARIO (mm)',
    'TEMPERATURA DO AR - BULBO SECO, HORARIA (C)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)'
]

# dropping: pressure, radiation, dew point, wind direction, wind speed - not relevant

chunks = []

for chunk in pd.read_csv('automatic_weather_stations_inmet_brazil_2000_2021.csv',
                         sep=';',
                         usecols=cols_we_want,
                         chunksize=500000,
                         low_memory=False):

    chunk.columns = ['station_id', 'date', 'precip_mm', 'temp_c', 'humidity_pct']

    # filter to 2016-2018 straight away so we dont store useless rows
    chunk['date'] = pd.to_datetime(chunk['date'], errors='coerce')
    chunk = chunk[chunk['date'] >= '2016-01-01']
    chunk = chunk[chunk['date'] < '2019-01-01']

    if len(chunk) > 0:
        chunks.append(chunk)

weather = pd.concat(chunks, ignore_index=True)
print(weather.shape)

(14276088, 5)


In [25]:
# make sure the number columns are actually numbers
weather['temp_c'] = pd.to_numeric(weather['temp_c'], errors='coerce')
weather['precip_mm'] = pd.to_numeric(weather['precip_mm'], errors='coerce')
weather['humidity_pct'] = pd.to_numeric(weather['humidity_pct'], errors='coerce')

# -9999 is used for missing values in this dataset, replace with NaN
weather = weather.replace(-9999.0, np.nan)

# group hourly into daily - average temp and humidity, total rain
weather_daily = weather.groupby(['station_id', 'date']).agg(
    temp_mean_c     = ('temp_c',       'mean'),
    temp_max_c      = ('temp_c',       'max'),
    temp_min_c      = ('temp_c',       'min'),
    precip_total_mm = ('precip_mm',    'sum'),
    humidity_mean   = ('humidity_pct', 'mean')
).reset_index()

# drop rows where all weather values are missing
weather_daily = weather_daily.dropna(subset=['temp_mean_c', 'precip_total_mm', 'humidity_mean'], how='all')

print(weather_daily.shape)
print(weather_daily.isnull().sum())

(594837, 7)
station_id             0
date                   0
temp_mean_c        39906
temp_max_c         39906
temp_min_c         39906
precip_total_mm        0
humidity_mean      47120
dtype: int64


In [26]:
weather_daily.to_csv('cleaned/weather_daily.csv', index=False)
print('done')

done


## quick check on all the cleaned files

note: sellers dataset was dropped completely - we only care about buyers not sellers
note: reviews dataset was also dropped for now - might use it later for sentiment

In [27]:
for fname in sorted(os.listdir('cleaned')):
    df = pd.read_csv('cleaned/' + fname)
    nulls = round(df.isnull().mean().mean() * 100, 1)
    print(fname, '|', df.shape, '| nulls:', str(nulls) + '%')

customers.csv | (99441, 5) | nulls: 0.0%
geolocation.csv | (14837, 4) | nulls: 0.0%
order_category.csv | (97277, 2) | nulls: 0.0%


order_payments.csv | (99440, 4) | nulls: 0.0%
order_spend.csv | (98666, 4) | nulls: 0.0%


orders.csv | (99441, 7) | nulls: 0.0%
products.csv | (32341, 2) | nulls: 0.0%


weather_daily.csv | (594837, 7) | nulls: 4.0%
weather_stations.csv | (611, 4) | nulls: 0.0%
